In [ ]:
# Databricks notebook source
# tutorial_name: 01-Day9-Monitoring-Automation-System-Tables
# notebookName: 01-Day9-Monitoring-Automation-System-Tables
# COMMAND ----------

# DBTITLE 1,Paths (same pattern as Days 5–9)
notepoint_rel = "hands-on/day-09/notebooks/01-Day9-Monitoring-Automation-System-Tables.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
DAY9_ROOT = f"{BASE_PATH}day09-{STUDENT_ID}"
GOLD_DAY8 = f"{DAY8_ROOT}/medallion/gold_by_destination"
# COMMAND ----------

# DBTITLE 1,Prerequisite check
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
print("DAY9_ROOT:", DAY9_ROOT)
# COMMAND ----------


# Day 9 — Item 23: Monitoring & Automation

Syllabus item 23 — job monitoring, **system tables**, **alerts**, **Jobs API**, **CI/CD** overview.

Focus: concepts + **diagrams** + **practical** Spark SQL checks you can run on a **cluster**. Some features (full **Jobs REST** from Python, **account-level** billing) need **admin tokens** or the **workspace UI** — those steps are spelled out in markdown for manual follow-through.

**Extra bundle pointers:** `databricks/**/Job_Scheduling_and_Monitoring*`, `**/Monitoring_Optimizing_Pipeline_Performance*`, Day 4 `system.access.audit` patterns.


## Student flow — item 23 (follow in order)

1. Run the **paths** cell and prerequisite check; fix **`P_BASIC`** if needed.
2. Run **cluster SQL / Python cells** in order; many use `try`/`except` because **`system.*`** entitlements vary by workspace.
3. When markdown refers to the **Jobs** UI, switch to the browser: **Workflows** → **Jobs** → inspect **Day 8** runs there (not inside a notebook cell).
4. Treat **Jobs API** and **CI/CD** sections as **read-only** unless the instructor gives you a token and permission to call APIs.



## What “monitoring” means here

- **Runs and logs:** Job and notebook runs leave history in **Workflows**; driver logs explain failures.
- **Data plane:** Clusters read and write **Delta on ABFS** as before.
- **Platform signals:** Where allowed, **`system.*`** SQL and **alerts** add operational views. **Jobs API** and **IaC** automate the same objects you see in the UI.

```mermaid
flowchart TB
  jobs[Job runs UI + logs]
  data[Delta on ABFS]
  sys[System tables SQL]
  alt[Alerts]
  data --> jobs
  jobs --> sys
  jobs --> alt
```


## Job monitoring (manual checklist — UI)

1. **Workflows** → **Jobs** → pick a job from **Day 8**.
2. Open **Runs** → select a run → **Tasks** tab: duration, **retry** count, **cluster** id.
3. Download **driver logs** for failures (Executor tab for distributed issues).
4. Compare **start/end** timestamps with **SLA** you promised the business.

Note: run a Day 8 job **on purpose** with a failing task (clone notebook with bad path) to see **Failed** state, then fix and re-run.


In [ ]:
# Exercise: Spark session identifiers (paste into support tickets)
import pyspark
print("Spark:", spark.version)
try:
    print("App id:", spark.sparkContext.applicationId)
except Exception as e:
    print("Context:", e)


## System tables & `system` catalog (concept)

Unity Catalog and workspace **system** objects expose **read-only** SQL for audit, lineage (where entitled), billing previews, etc. Names and availability change by **region** and **entitlements**.

**Related to Day 4:** `system.access.audit`, `system.information_schema.catalogs` — here the emphasis is on day-to-day operations and job health.


## Figure — who can query what

```mermaid
flowchart LR
  U[User / Service principal]
  U -->|SELECT| ST[System tables]
  A[Account admin] -->|more views| ST
  W[Workspace admin] -->|audit config| CFG[Diagnostic settings]
```

If `SELECT` is denied, use **workspace UI** job history or **Azure Monitor** / **Diagnostic logs** (platform admin).


In [ ]:
# Exercise: system.access.audit sample (often restricted — same as Day 4)
try:
    spark.sql("SELECT event_time, action_name, user_identity.email AS email FROM system.access.audit ORDER BY event_time DESC LIMIT 10").show(truncate=False)
except Exception as e:
    print("audit query not available:", type(e).__name__, str(e)[:200])


In [ ]:
# Exercise: UC information_schema catalogs (lighter privilege than audit)
try:
    spark.sql("SELECT catalog_name FROM system.information_schema.catalogs LIMIT 15").show(truncate=False)
except Exception as e:
    print("catalogs:", type(e).__name__, str(e)[:200])


## Alerts (concept + manual steps)

**Job alerts:** Job settings → **Notifications** → failure / timeout / long run.

**SQL alerts:** Databricks SQL → **Alert** on a query (see notebook **02**).

**Diagram**

```text
  Condition (failed run | query threshold)
           |
           v
  Notification channel (email, Teams, PagerDuty via webhook)
```


## Jobs API & automation (no secrets in the notebook)

Typical flow:

1. Create a **PAT** or **OAuth** for a **service principal** with **CAN MANAGE RUN** on the job.
2. Call **`GET /api/2.1/jobs/runs/list`** or **`jobs/run-now`** with `curl` or **Python `requests`** from a **secure** runner (not committed code).
3. Store JSON job definitions in **Git**; apply with **Terraform** `databricks_job` or **Databricks CLI** `bundle deploy`.

**CI/CD overview (course level):** build → test → deploy job JSON → **Run now** in staging → promote. Details: `databricks/**/15-Automating-Jobs-with-CI-CD-Pipelines*`.


In [ ]:
# Exercise: build a Jobs API URL template (fill host + token outside the notebook)
WORKSPACE_HOST = "https://YOUR-WORKSPACE.cloud.databricks.com"  # or adb-*.azuredatabricks.net
print("Example list runs:")
print(f"{WORKSPACE_HOST}/api/2.1/jobs/runs/list?limit=5")
print("Use: curl -H \"Authorization: Bearer $TOKEN\" \"...\" ")


## Data observability — Delta on ABFS (practical)

Even without system tables, you can **monitor data** the same way as **Day 5–6**:

- `DESCRIBE HISTORY`, `DESCRIBE DETAIL`
- Row counts vs **yesterday**
- **File counts** growth under the path


In [ ]:
# Exercise: Delta history on P_BASIC (production-minded check)
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "operation", "timestamp").show(8, truncate=False)


In [ ]:
# Exercise: optional read Day 8 Gold if you ran Day 8 pipeline
try:
    spark.read.format("delta").load(GOLD_DAY8).orderBy("total_flights", ascending=False).show(5, truncate=False)
except Exception as e:
    print("Day 8 Gold not present yet (run Day 8 notebook 02 first):", type(e).__name__)


## Operational checklist (takeaway)

| Layer | Check |
|-------|--------|
| Job | Last run success, duration trend |
| Cluster | Autoscaling bounds, DBR version |
| Data | Delta history, schema drift |
| Security | Audit who touched `system` or storage |
| Cost | Warehouse auto-stop (notebook **02**), job cluster policy |


## Next notebook

Open **`02-Day9-Databricks-SQL-Warehouses-Dashboards.ipynb`** for item **24** (SQL warehouses, dashboards, alerts in **Databricks SQL**).


In [ ]:
# Optional cleanup
# dbutils.fs.rm(DAY9_ROOT, recurse=True)


## Figure — alert paths (jobs + data)

```mermaid
flowchart LR
  JR[Job run failed]
  JR --> N1[Email / Teams]
  DQ[SQL alert query breach]
  DQ --> N2[Webhook]
  ST[System table rule]
  ST --> N3[SIEM export]
```


## Manual deep-dive — Azure Monitor (platform)

If your org streams Databricks **diagnostic logs** to **Log Analytics**, platform admins can build **KQL** dashboards for **job failures**, **cluster** creation, **notebook** access. This is **outside** the notebook; ask your admin for a **saved workbook** link.


In [ ]:
# Exercise: row-level sanity on P_BASIC (monitoring data not just jobs)
spark.read.format("delta").load(P_BASIC).groupBy("DEST_COUNTRY_NAME").count().orderBy("count", ascending=False).limit(5).show(truncate=False)


## Tie-back — Day 4 security notebook

Re-open **`03-Day4-Unity-Catalog-Security-Lineage.ipynb`** section on **`system.access.audit`** — Day 9 applies that **audit** data to **operations** (who changed what, when).


## Figure — CI/CD in one picture

```mermaid
flowchart LR
  G[Git repo]
  G --> P[CI pipeline]
  P --> T[Tests]
  T --> D[Deploy job JSON]
  D --> W[Databricks workspace]
```


In [ ]:
# Exercise: log a string you would send to your CI system (no network call)
print("Example: POST job definition to workspace after tests pass.")
